# NHS Workforce Hierarchical Forecast: Interactive Dashboard
An interactive Dash app for Trust, Service, Staff Group, and Metric-level exploration of observed and forecasted NHS workforce trends using hierarchical time series models.


In [1]:
import os
import dash
from dash import dcc, html, Output, Input
import pandas as pd
import plotly.graph_objects as go 

## 1. 🏗️ Load Data and Setup
- Loads NHS workforce forecasts + actuals (combined, “long” tidy table) prepped for all groups and methods.
- Columns are pre-split for easy drop-down filtering.
- Data source: `nhs_hierarchical_forecasts_long1.csv` (output from hierarchy, forecast, and merge workflow).

In [2]:
base_dir = r"C:\Users\Destinee.HassanBien\Documents\Desys Work"
csv_file = os.path.join(base_dir, "nhs_hierarchical_forecasts_long1.csv")
df = pd.read_csv(csv_file, parse_dates=["ds"])

In [3]:
service_groups = sorted(df['service_group'].dropna().unique())
staff_groups = sorted(df['staff_group'].dropna().unique())
metrics = sorted(df['metric'].dropna().unique())
forecast_cols = ['Naive', 'Naive/BottomUp', 'Naive/MinTrace_method-ols', 'Naive/MinTrace_method-mint_shrink']

## 2. 🎛 Dashboard Layout: Dropdowns, Graph, Summary
How Dashboard Filtering Works
- Four dropdowns: Service Group, Staff Group, Metric, and Forecast Methods.
- Each filter is dynamically updated to prevent empty selections.
- Users can select any valid combination to drill down to a specific NHS group/metric and compare observed vs multiple forecasts.

In [5]:
app = dash.Dash(__name__)
app.layout = html.Div([
    html.H1("NHS Hierarchical Workforce Forecast Dashboard"),
    html.Div([
        html.Label("Service Group:"),
        dcc.Dropdown(
            id='service-group',
            options=[{'label': sg, 'value': sg} for sg in service_groups],
            value=service_groups [0] if service_groups else None
        ),
        html.Label("Staff Group:"),
        dcc.Dropdown(
            id='staff-group',
            options=[],
            value=None
        ),
        html.Label("Metric:"),
        dcc.Dropdown(
            id='metric',
            options=[],
            value=None
        ),
        html.Label("Forecast Methods:"),
        dcc.Dropdown(
            id='forecast-cols',
            options=[{'label': c, 'value': c} for c in forecast_cols],
            value=['Naive', 'Naive/BottomUp'],
            multi=True
        ),
    ], style={'width': '45%', 'display': 'inline-block', 'vertical-align': 'top', 'padding': '10px'}),
    dcc.Graph(id='ts-plot'),
    html.Div(id='summary')
])

## 3. 🔁 Dash Callbacks: Filter Logic
- Each dropdown’s options update based on the parent filter’s selection. E.g., selecting a Service Group limits Staff Groups, then Metrics, etc.
- Avoids empty graphs and disables impossible combinations!

In [6]:
# Callbacks for filtering 
@app.callback(
    Output('staff-group', 'options'),
    Output('staff-group', 'value'),
    Input('service-group', 'value')
)
def update_staff_Group(service_group):
    sgs= df[df['service_group'] == service_group]['staff_group'].dropna().unique()
    options = [{'label': sg, 'value': sg} for sg in sorted(sgs)]
    return options, options[0]['value'] if options else None

@app.callback(
     Output('metric', 'options'),
    Output('metric', 'value'),
    Input('service-group', 'value'),
    Input('staff-group', 'value')
)
def update_metric(service_group, staff_group):
    mtx = df[(df['service_group'] == service_group) & (df['staff_group'] == staff_group)]['metric'].dropna().unique()
    options = [{'label': m, 'value': m} for m in sorted(mtx)]
    return options, options[0]['value'] if options else None

@app.callback(
    Output('ts-plot', 'figure'),
    Output('summary', 'children'),
    [Input('service-group', 'value'),
     Input('staff-group', 'value'),
     Input('metric', 'value'),
     Input('forecast-cols', 'value')]
)
def update_plot(service_group, staff_group, metric, selected_fc):
    group_df = df[
        (df['service_group'] == service_group) &
        (df['staff_group'] == staff_group) & 
        (df['metric'] == metric)].sort_values('ds')
    fig = go.Figure()
    if not group_df.empty:
        fig.add_trace(go.Scatter(
            x=group_df['ds'], y=group_df['observed'],
            mode='lines+markers', name='Observed', line=dict(color='black', width=2)
        ))
        color_map = {'Naive':'#1f77b4', 'Naive/BottomUp':'#ff7f0e',
                     'Naive/MinTrace_method-ols':'#d62728','Naive/MinTrace_method-mint_shrink':'#2ca02c'}
        for col in selected_fc:
            if col in group_df.columns:
                fig.add_trace(go.Scatter(
                    x=group_df['ds'], y=group_df[col],
                    mode='lines+markers', name=col,
                    line=dict(color=color_map.get(col, None))
                ))
        last_obs = group_df[group_df['observed'].notnull()]['ds'].max()
        if pd.notnull(last_obs):
            fig.add_vline(x=last_obs, line=dict(dash='dash', color='gray'))
        fig.update_layout(
            title=f"{service_group} / {staff_group} / {metric}",
            xaxis_title="Date", yaxis_title="Monthly Count", hovermode='x unified'
        )
    summary = f"{group_df['observed'].count()} observed months; forecast to {group_df['ds'].max().strftime('%b %y') if not group_df.empty else 'N/A'}"
    return fig, summary 

## 4. 📈 Plot: Observed and Forecasted Series
Interpretation
- **Black Line:** Observed (historic) values for selected group/metric.
- **Colored Lines:** Model forecasts (Naive, BottomUp, MinT-OLS, MinT-shrink).
- Dashed vertical line marks forecast start (last date with real observed value).
- Plot allows instant visual benchmarking of forecast reliability for operational, planning, and QA use.

In [7]:
if __name__ == "__main__":
    app.run(debug=True)

## 6. ⚙️ Deployment Note
- Place all notebook/code/markdown and CSV files in the correct directory for full app function.
- For Jupyter Book: include a note on “how to launch this app” (either from terminal, JupyterLab, or via NHS intranet link if deployed).

## 7. ✔️ Next Steps / To Do
To Do
- [ ] Review/dropdown options to confirm all expected NHS groups appear.
- [ ] QA the exported CSV that feeds this app—no duplicate months, all needed groups/levels.
- [ ] Extend summary metrics (e.g., add RMSE, best-model) and add benchmark horizontal lines if needed for dashboards.
- [ ] Provide training/readme for NHS operational users or analysts.